# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides you in loading, exploring, and processing the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/api.html) library, based on a Croissant schema and referencing all entities by their `@id`.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install -U mlcroissant

## 1. Data Loading
Load FAIR^2 dataset metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s. For reference, use the `record_set` attribute in Croissant metadata, and enumerate their `field` definitions by `@id`.

**Note**: All entities are referenced by their `@id` for clarity and reproducibility.

In [ ]:
# List all available record sets by their @id
if hasattr(md, 'record_set') and md.record_set:
    for rs in md.record_set:
        print(f"RecordSet @id: {rs['@id']}")
        # Show fields in each RecordSet
        if 'field' in rs and rs['field']:
            print("  Fields:")
            for fld in rs['field']:
                print(f"    {fld['@id']}")
        print()
else:
    print("No record sets defined in schema metadata.")

## 3. Data Extraction

Let's extract all data for each RecordSet by its `@id`, loading the records into pandas DataFrames. Use the unique RecordSet and Field `@id`s from the overview above.

In [ ]:
# Collect all record set @ids
record_sets_ids = []
if hasattr(md, 'record_set') and md.record_set:
    record_sets_ids = [rs['@id'] for rs in md.record_set]
else:
    print("No record sets defined.")

dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns of first record set as an example
if dataframes:
    first_record_set_id = record_sets_ids[0]
    print(f"\nAvailable columns in RecordSet {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing on a numerical or key categorical field. 
This will demonstrate:
- Filtering records (e.g. Age > 60)
- Normalizing a numeric field
- Grouping by best-guess categorical field (e.g., Sex, msi_status)

Replace `<numeric_field_id>` and `<group_field_id>` with available IDs from above as appropriate for your dataset.

In [ ]:
# Pick appropriate IDs based on the schema's fields
record_set_id = None
numeric_field_id = None
group_field_id = None

# Attempt to auto-pick suitable fields
for rs in (md.record_set or []):
    # Try to find a field matching usual numeric variable names
    fields = rs.get('field', [])
    candidate_numeric = [f['@id'] for f in fields if 'age' in f['@id'].lower() or ('number' in f.get('dataType','').lower())]
    candidate_group = [f['@id'] for f in fields if any(k in f['@id'].lower() for k in ['sex','msi','group','anatom','status'])]
    if candidate_numeric:
        numeric_field_id = candidate_numeric[0]  # e.g., 'age' or similar
        record_set_id = rs['@id']
    if candidate_group:
        group_field_id = candidate_group[0]
    if record_set_id and numeric_field_id:
        break

if not (record_set_id and numeric_field_id):
    # Fallback to first available if not found
    if dataframes and record_sets_ids:
        record_set_id = record_sets_ids[0]
        numeric_field_id = dataframes[record_set_id].columns[0] if len(dataframes[record_set_id].columns) > 0 else None

df = dataframes.get(record_set_id)
if df is None or numeric_field_id is None:
    print("Appropriate DataFrame or numeric field not found.")
else:
    # Try to convert to numeric, if possible
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.75) if not df[numeric_field_id].isnull().all() else 0
    filtered_df = df[df[numeric_field_id] > threshold] if threshold else df
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalizing the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id if available
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped)
    else:
        print("\nNo categorical grouping field found for this record set.")

## 5. Visualization

Visualize a numeric field distribution (e.g. age or interval) and optionally relationships between a numeric and a categorical field using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group field exists, show boxplot
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

- This notebook demonstrated how to load and explore the FAIR^2 dataset using the `mlcroissant` library.
- We showed how to reference data schema entities strictly via their `@id` fields for reproducibility.
- Through filtering, normalization, grouping, and visualization, you can now proceed to domain-specific analyses for clinicopathological or biomarker-driven investigations among second primary colorectal cancer survivors.

For more details, refer to the dataset's full metadata and schema documentation at [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and see the [`mlcroissant` documentation](https://mlcommons.github.io/croissant/api.html).